# Week 1: Prompt Engineering and System Prompts
Define the strict system prompts for the **Writer** and **Critic** agents.
Experiment with few-shot prompting to establish tone and structure.

In [ ]:
!pip install -q langchain langchain-openai langchain-anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Load API keys from your .env file or environment
load_dotenv()

import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# Using GPT-4o for code generation and document analysis tasks
llm = ChatOpenAI(model="gpt-4o", temperature=0)

### 1. The Writer Agent Prompt
Responsible for drafting the document from rough notes. We define a few-shot prompt to guide its behavior.

In [ ]:
writer_system_prompt = """
You are an expert Document Writer AI specialized in creating highly structured, professional Product Requirement Documents (PRDs).
Your task is to transform raw meeting notes or bullet points into a well-formatted Markdown PRD.

A standard PRD must include the following sections:
1. Executive Summary
2. Business Objectives & Success Metrics
3. User Personas
4. Features and Requirements (User Stories)
5. Edge Cases & Technical Specs
"""

writer_prompt_template = ChatPromptTemplate.from_messages([
    ("system", writer_system_prompt),
    ("user", "Here are the raw notes: {raw_notes}")
])

writer_chain = writer_prompt_template | llm

In [ ]:
rough_notes = """
- new feature: user profile picture upload
- need to accept PNG, JPG
- max size 5MB
- should automatically crop to square
"""

# test the writer
try:
    draft = writer_chain.invoke({"raw_notes": rough_notes})
    print(draft.content)
except Exception as e:
    print("API Error:", e)

### 2. The Critic Agent Prompt
Responsible for reviewing the draft and returning structured feedback or a missing section flag.

In [ ]:
critic_system_prompt = """
You are a strict QA Compliance / Reviewer AI.
Your objective is to evaluate the provided document draft against the standard template.

Review Rubric:
- Does it contain 'Business Objectives & Success Metrics'?
- Does it contain 'User Personas'?
- Are there clear 'Edge Cases'?

If the draft is missing any of these elements, reply with specific feedback stating what is missing.
If it is complete and meets all standards, reply with 'APPROVED'.
"""

critic_prompt_template = ChatPromptTemplate.from_messages([
    ("system", critic_system_prompt),
    ("user", "Draft for review:\n\n{draft}")
])

critic_chain = critic_prompt_template | llm

In [ ]:
# test the critic against our generated draft
try:
    feedback = critic_chain.invoke({"draft": draft.content})
    print(feedback.content)
except Exception as e:
    print("API Error:", e)